In [10]:
from nilearn import datasets
from src.utils import *
from nilearn import image 
import numpy as np
import pandas as pd

In [19]:
def atlas_overlap(nifti_file:str, parcel_num:int, atlas_string:str, top_10:bool=True):
    """
    Create a dataframe with the labels of regions and number of voxels that overlap across the given parcel number and atlas/parcellation. 
    
    Parameters:
    nifti_file : str or pathlib.Path
        Path to the labeled parcellation NIfTI file.
    parcel_num : int
        Label of the parcel to extract. Must correspond to a valid
        non-background parcel (background is label 0).
    atlas_string : str
        Name of atlas 
    top_10 : bool
        If True filters to show only the top 10 corresponding parcels. Otherwise, shows all parcels.
    
    Returns a dataframe containing columns:
        regions = Regional labels for the given atlas/parcellation specified in the atlas_string input, ordered by overlap.
        intersection_voxels = Number of voxels that overlap across the neurosynth parcel and the given atlas/parcellation specified in the atlas_string input.
        parcel_overlap = Percentage of voxels in the neurosynth parcellation that overlap with the given atlas/parcellation specified in the atlas_string input.

    """
    atlas = datasets.fetch_atlas_schaefer_2018()
    parcellation_file = nifti_file
    _, mask_img = extract_parcel_mask(parcellation_file, 3)

    atlas_img = image.resample_to_img(
        atlas.maps,
        mask_img, 
        interpolation="nearest",
    )

    parcel = mask_img.get_fdata().astype(bool)
    atlas_data = atlas_img.get_fdata().astype(int)

    results = []
    
    for region_id, region_name in enumerate(atlas.labels[1:], start=1):
        region_mask = atlas_data == region_id
        intersection = np.logical_and(parcel, region_mask).sum()

        results.append({
            "regions": region_name,
            "intersection_voxels": intersection,
            "parcel_overlap": intersection / parcel.sum()
        })
    overlap_df = (
        pd.DataFrame(results)
        .sort_values("parcel_overlap", ascending=False)
        .reset_index(drop=True)
    )
    overlap_df2 = overlap_df[overlap_df["intersection_voxels"] > 0] 
    if top_10: 
        overlap_df2 = overlap_df2.iloc[0:10]
    return overlap_df2

In [18]:
atlas_overlap("/Users/miacasburn/Documents/GitHub/neurolabel/parcellations/neurosynth/Neurosynth_Parcellation_k200.nii.gz", 3, " ") 

[fetch_atlas_schaefer_2018] Dataset found in /Users/miacasburn/nilearn_data/schaefer_2018

,regions,intersection_voxels,parcel_overlap
0,7Networks_LH_Default_Par_2,278,0.240901
1,7Networks_RH_Default_Par_3,189,0.163778
2,7Networks_RH_Default_Par_2,169,0.146447
3,7Networks_RH_DorsAttn_Post_3,145,0.125650
4,7Networks_LH_DorsAttn_Post_4,97,0.084055
5,7Networks_LH_Default_Par_5,73,0.063258
6,7Networks_LH_SalVentAttn_TempOcc_1,43,0.037262
7,7Networks_LH_Default_Par_1,28,0.024263
8,7Networks_LH_Default_Par_4,27,0.023397
9,7Networks_RH_Default_Par_1,17,0.014731
